# berts-gg-qa-training

Training-only notebook (internet OFF) for Google QUEST.

- Ensemble backbones: RoBERTa-base, RoBERTa-large, DeBERTa-v3-base
- Dual-tower regression model
- Mixed precision + optional SWA
- No KFold/skfold
- Output: 3 SWA model weights + artifacts in `/kaggle/working/ggqa_bert_models`


In [ ]:
import os, gc, re, json, html, random
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup


In [ ]:
@dataclass
class CFG:
    seed: int = 42
    epochs: int = 3
    train_bs: int = 4
    valid_bs: int = 8
    grad_accum: int = 2
    lr: float = 2e-5
    wd: float = 0.01
    max_len_q: int = 256
    max_len_a: int = 256
    num_workers: int = 2
    use_amp: bool = True
    use_swa: bool = True
    swa_start_epoch: int = 1
    out_dir: str = '/kaggle/working/ggqa_bert_models'

    model_zoo = {
        'roberta_base': '/kaggle/input/datasets/abhishek/roberta-base',
        'roberta_large': '/kaggle/input/datasets/marshal02/robertalarge',
        'deberta_v3_base': '/kaggle/input/datasets/debarshichanda/debertav3base'
    }

cfg = CFG()
os.makedirs(cfg.out_dir, exist_ok=True)
os.makedirs(f"{cfg.out_dir}/artifacts", exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


In [ ]:
# preprocessing adapted from public kernels
puncts = [',', '.', '"', ':', ')', '(', '-', '!', '?', '|', ';', "'", '$', '&', '/', '[', ']', '>', '%', '=', '#', '*', '+', '\\', '\\n', '\\t']
mispell_dict = {
    "can't": "cannot", "won't": "will not", "don't": "do not",
    "doesn't": "does not", "didn't": "did not", "it's": "it is",
    "i'm": "i am", "you're": "you are", "they're": "they are"
}

def clean_text(x):
    x = str(x)
    for punct in puncts:
        x = x.replace(punct, f' {punct} ')
    return x

def clean_numbers(x):
    x = re.sub('[0-9]{5,}', '#####', x)
    x = re.sub('[0-9]{4}', '####', x)
    x = re.sub('[0-9]{3}', '###', x)
    x = re.sub('[0-9]{2}', '##', x)
    return x

def replace_typical_misspell(text):
    text = str(text)
    for k, v in mispell_dict.items():
        text = text.replace(k, v)
    return text

def clean_data(df, columns):
    out = df.copy()
    for c in columns:
        out[c] = out[c].fillna('').apply(lambda x: html.unescape(str(x)))
        out[c] = out[c].apply(clean_numbers).str.lower().apply(clean_text).apply(replace_typical_misspell)
    return out


In [ ]:
train = pd.read_csv('/kaggle/input/competitions/google-quest-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/google-quest-challenge/test.csv')

text_cols = ['question_title','question_body','answer']
train = clean_data(train, text_cols)
test = clean_data(test, text_cols)

target_cols = [c for c in train.columns if c not in test.columns and c != 'qa_id']
y = train[target_cols].values.astype(np.float32)
label_values = {c: np.sort(train[c].unique()) for c in target_cols}
groups = train['question_body'].values

print('targets:', len(target_cols))


In [ ]:
def mean_spearman(y_true, y_pred):
    vals=[]
    for i in range(y_true.shape[1]):
        c = spearmanr(y_true[:, i], y_pred[:, i]).correlation
        vals.append(0.0 if np.isnan(c) else c)
    return float(np.mean(vals))

CLIPPINGS = {
    'question_has_commonly_accepted_answer': (0.0, 0.6),
    'question_conversational': (0.15, 1.0),
    'question_multi_intent': (0.1, 1.0),
    'question_type_choice': (0.1, 1.0),
    'question_type_compare': (0.1, 1.0),
    'question_type_consequence': (0.08, 1.0),
    'question_type_definition': (0.1, 1.0),
    'question_type_entity': (0.13, 1.0),
}

def apply_clipping(preds, target_cols):
    z = preds.copy()
    for j,c in enumerate(target_cols):
        if c in CLIPPINGS:
            lo,hi = CLIPPINGS[c]
            z[:,j] = np.clip(z[:,j], lo, hi)
    return z

def snap_to_known_labels(preds, target_cols, label_values):
    out = np.zeros_like(preds)
    for j,c in enumerate(target_cols):
        vals = np.asarray(label_values[c], dtype=np.float32)
        idx = np.abs(preds[:, [j]] - vals.reshape(1, -1)).argmin(axis=1)
        out[:,j] = vals[idx]
    return out

def ensure_not_constant(preds):
    out = preds.copy()
    for j in range(out.shape[1]):
        if np.allclose(out[:,j], out[0,j]):
            out[:,j] = out[:,j] + np.linspace(0, 1e-7, out.shape[0], dtype=np.float32)
    return out


In [ ]:
class QADataset(Dataset):
    def __init__(self, df, labels, tokenizer, max_len_q, max_len_a):
        self.df = df.reset_index(drop=True)
        self.labels = labels
        self.tok = tokenizer
        self.max_len_q = max_len_q
        self.max_len_a = max_len_a

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        qtxt = f"{r['question_title']} [SEP] {r['question_body']}"
        atxt = f"{r['question_title']} [SEP] {r['answer']}"

        q = self.tok(qtxt, truncation=True, max_length=self.max_len_q, padding='max_length', return_tensors='pt')
        a = self.tok(atxt, truncation=True, max_length=self.max_len_a, padding='max_length', return_tensors='pt')

        d = {
            'q_input_ids': q['input_ids'][0],
            'q_attention_mask': q['attention_mask'][0],
            'a_input_ids': a['input_ids'][0],
            'a_attention_mask': a['attention_mask'][0],
        }
        if self.labels is not None:
            d['labels'] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return d

class DualTowerRegressor(nn.Module):
    def __init__(self, model_path, n_targets):
        super().__init__()
        self.q_encoder = AutoModel.from_pretrained(model_path, local_files_only=True)
        self.a_encoder = AutoModel.from_pretrained(model_path, local_files_only=True)
        hidden = self.q_encoder.config.hidden_size
        self.drop = nn.Dropout(0.2)
        self.head = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.GELU(), nn.Linear(hidden, n_targets))

    def encode(self, encoder, ids, mask):
        out = encoder(ids, attention_mask=mask).last_hidden_state
        m = mask.unsqueeze(-1)
        return (out * m).sum(1) / m.sum(1).clamp(min=1)

    def forward(self, q_ids, q_mask, a_ids, a_mask):
        qv = self.encode(self.q_encoder, q_ids, q_mask)
        av = self.encode(self.a_encoder, a_ids, a_mask)
        return self.head(self.drop(torch.cat([qv, av], dim=1)))


In [ ]:
# deterministic single split by groups (no KFold)
unique_groups = pd.Series(groups).drop_duplicates().values
rng = np.random.default_rng(cfg.seed)
shuffled = rng.permutation(unique_groups)
val_size = max(1, int(0.1 * len(shuffled)))
val_groups = set(shuffled[:val_size])
va_idx = np.where(pd.Series(groups).isin(val_groups).values)[0]
tr_idx = np.where(~pd.Series(groups).isin(val_groups).values)[0]
print('train/val:', len(tr_idx), len(va_idx))


In [ ]:
def predict_loader(model, loader):
    model.eval()
    ps=[]
    with torch.no_grad():
        for b in loader:
            p = model(
                b['q_input_ids'].to(device), b['q_attention_mask'].to(device),
                b['a_input_ids'].to(device), b['a_attention_mask'].to(device)
            ).detach().cpu().numpy()
            ps.append(p)
    return np.concatenate(ps, axis=0)

def train_one_model(model_name, model_path):
    tok = AutoTokenizer.from_pretrained(model_path, local_files_only=True)

    tr_ds = QADataset(train.iloc[tr_idx], y[tr_idx], tok, cfg.max_len_q, cfg.max_len_a)
    va_ds = QADataset(train.iloc[va_idx], y[va_idx], tok, cfg.max_len_q, cfg.max_len_a)
    te_ds = QADataset(test, None, tok, cfg.max_len_q, cfg.max_len_a)

    tr_ld = DataLoader(tr_ds, batch_size=cfg.train_bs, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
    va_ld = DataLoader(va_ds, batch_size=cfg.valid_bs, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
    te_ld = DataLoader(te_ds, batch_size=cfg.valid_bs, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)

    model = DualTowerRegressor(model_path, len(target_cols)).to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
    total_steps = max(1, len(tr_ld) * cfg.epochs // cfg.grad_accum)
    sched = get_linear_schedule_with_warmup(optim, int(0.1 * total_steps), total_steps)
    scaler = GradScaler(enabled=cfg.use_amp)
    crit = nn.SmoothL1Loss()

    best_cv = -1
    best_state = None
    swa_state = None

    for ep in range(cfg.epochs):
        model.train()
        optim.zero_grad(set_to_none=True)

        for step, b in enumerate(tr_ld):
            with autocast(enabled=cfg.use_amp):
                pred = model(
                    b['q_input_ids'].to(device), b['q_attention_mask'].to(device),
                    b['a_input_ids'].to(device), b['a_attention_mask'].to(device)
                )
                loss = crit(pred, b['labels'].to(device)) / cfg.grad_accum

            scaler.scale(loss).backward()
            if (step + 1) % cfg.grad_accum == 0:
                scaler.step(optim); scaler.update(); optim.zero_grad(set_to_none=True); sched.step()

        val_pred = predict_loader(model, va_ld)
        cv = mean_spearman(y[va_idx], val_pred)
        print(model_name, 'epoch', ep, 'cv', cv)

        if cv > best_cv:
            best_cv = cv
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if cfg.use_swa and ep >= cfg.swa_start_epoch:
            cur = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if swa_state is None:
                swa_state = cur
            else:
                for k in swa_state:
                    swa_state[k] = 0.5 * swa_state[k] + 0.5 * cur[k]

    final_state = swa_state if (cfg.use_swa and swa_state is not None) else best_state
    save_path = f"{cfg.out_dir}/{model_name}_swa.pt"
    torch.save(final_state, save_path)

    model.load_state_dict(final_state)
    val_pred = predict_loader(model, va_ld)
    test_pred = predict_loader(model, te_ld)

    del model, tr_ds, va_ds, te_ds, tr_ld, va_ld, te_ld
    gc.collect(); torch.cuda.empty_cache()
    return val_pred, test_pred, best_cv, save_path


In [ ]:
all_val = {}
all_test = {}
model_paths = {}

for name, path in cfg.model_zoo.items():
    print('\n===', name, '===')
    va_pred, te_pred, score, sp = train_one_model(name, path)
    oof = np.zeros((len(train), len(target_cols)), dtype=np.float32)
    oof[va_idx] = va_pred
    all_val[name] = oof
    all_test[name] = te_pred
    model_paths[name] = sp
    print('saved:', sp, 'best_cv:', score)

ensemble_oof = np.mean(np.stack(list(all_val.values()), axis=0), axis=0)
ensemble_test = np.mean(np.stack(list(all_test.values()), axis=0), axis=0)
print('Ensemble CV (val):', mean_spearman(y[va_idx], ensemble_oof[va_idx]))


In [ ]:
pp_val = snap_to_known_labels(apply_clipping(ensemble_oof, target_cols), target_cols, label_values)
pp_test = snap_to_known_labels(apply_clipping(ensemble_test, target_cols), target_cols, label_values)
pp_val = ensure_not_constant(pp_val)
pp_test = ensure_not_constant(pp_test)

print('PP CV (val):', mean_spearman(y[va_idx], pp_val[va_idx]))

np.save(f"{cfg.out_dir}/artifacts/oof_ensemble.npy", ensemble_oof)
np.save(f"{cfg.out_dir}/artifacts/test_ensemble.npy", ensemble_test)
np.save(f"{cfg.out_dir}/artifacts/test_final_pp.npy", pp_test)

with open(f"{cfg.out_dir}/artifacts/target_cols.json", 'w') as f:
    json.dump(target_cols, f)
with open(f"{cfg.out_dir}/artifacts/label_values.json", 'w') as f:
    json.dump({k:[float(x) for x in v] for k,v in label_values.items()}, f)
with open(f"{cfg.out_dir}/artifacts/clippings.json", 'w') as f:
    json.dump({k:[float(v[0]), float(v[1])] for k,v in CLIPPINGS.items()}, f)
with open(f"{cfg.out_dir}/artifacts/model_manifest.json", 'w') as f:
    json.dump(model_paths, f, indent=2)

print('Training done. 3 model weights:', model_paths)
